# Let's reproduce GPT-2 (124M)（导读）

和 07 的关系：**架构基本没变，变的是工程。**

07 章那 224 行已经是一个正确的 Transformer 了。这一章做三件 07 没做的事：

1. **对齐真实的 GPT-2**——用 GPT-2 的 BPE 词表（50257）、可加载 OpenAI 的官方权重、
   逐项对上超参数
2. **把训练速度榨出来**——TF32 / bfloat16 / torch.compile / Flash Attention，
   累计约 **10 倍**加速，一行架构代码都不改
3. **上真实数据**——FineWeb-Edu 10B tokens，配 HellaSwag 评测

**参考实现在 `build-nanogpt/`**，它的 44 个 commit 就是视频进度条。


## 怎么用那三个仓库

### `build-nanogpt/` —— 这一章的主线，跟着它走

Karpathy 专为这个视频建的，**每个 commit 对应视频里的一步**：

```bash
cd build-nanogpt
git log --reverse --oneline        # 从头到尾，44 步
git checkout <hash>                # 跳到视频某一时刻的代码
git diff <hash1> <hash2>           # 看这一步到底改了什么
git checkout master                # 回到终态
```

卡住的时候，`git diff` 上一个 commit 比看视频回放快得多。

文件：
- `train_gpt2.py` —— 主文件，从头写到尾都在这一个文件里
- `fineweb.py` —— 下载 + 分词 FineWeb-Edu，存成 shard
- `hellaswag.py` —— 评测
- `play.ipynb` —— 视频里的零散实验

### `nanoGPT/` —— 成熟版，写完之后拿来对照

比 build-nanogpt 更工程化：`model.py` / `train.py` 分离，配置文件、checkpoint、
resume、DDP、多种数据集脚本都齐全。**不是这一章的教材**，是你自己写完之后
"生产上应该长什么样"的参照物。

值得读的：`model.py` 里 `from_pretrained` 怎么把 HF 权重映射过来、
`configure_optimizers` 怎么分 weight decay 组。

### `llm.c/` —— 同一个模型的纯 C/CUDA 实现（本仓库没克隆）

没有 PyTorch，手写前向和反向 kernel。**和这一章是平行关系，不是后续。**
它的价值是让你看见 `torch.compile` 和 Flash Attention 在底下究竟做了什么。
建议这一章跑完之后再看，需要就说一声，我 clone 下来。

### 你自己已有的三个目录（顺带一提）

| 目录 | 是什么 |
|---|---|
| `nnzh/` | **你自己写的**共享包，`pip install -e .` 装进了 nn 环境，所以任何目录都能 `import nnzh`。进版本控制。 |
| `nn-zero-to-hero/` | Karpathy 的**课程 notebook 仓库**（lecture 1-8 的 .ipynb）。上游 clone，只作对照，已 gitignore。 |
| `ng-video-lecture/` | Karpathy 为 **lecture 7**（Let's build GPT）建的小仓库，就是那个 `gpt.py`。同样是上游 clone。 |

三个都是本地 clone 或本地包，**不会被推到你的 GitHub**——它们各自带 `.git`，
硬塞进去只会留一个空的 gitlink。


## 0 — 先认清硬件现实

Karpathy 用 **8×A100 80GB**，10B tokens 跑约 1-2 小时（云上约 $10）。

你的是 **RTX 4060 8GB**。按算力比，同样的 10B tokens 大约要 **60-90 小时**不间断。
显存倒是够（124M 参数 + AdamW 状态约 2GB，靠梯度累积把 micro-batch 压到 4-8 就行），
**瓶颈是时间不是显存**。

所以务实的路线：

| 阶段 | 在哪跑 |
|---|---|
| 写代码、调通、tiny shakespeare 过拟合一个 batch | 4060，几分钟 |
| 速度优化逐项验证（TF32/bf16/compile/flash） | 4060，每项几分钟，**加速比看得很清楚** |
| 小规模真跑（1B tokens 左右） | 4060，通宵一晚 |
| 完整 10B tokens 复现 | 租卡（Lambda / vast.ai，$10-20） |

FineWeb-Edu 的 10B 采样分词后约 20GB shard，你还有 878GB，够。


## 1 — 对齐 GPT-2 的架构

从加载 OpenAI 官方 124M 权重开始——**先能跑通别人的权重，再训自己的**。
这一步能立刻验证你的模块名和形状是不是对的。

和 07 章的差异：

| | 07 章 | GPT-2 |
|---|---|---|
| 词表 | 65 字符 | 50257 BPE（tiktoken） |
| block_size | 256 | 1024 |
| 规模 | 384/6/6 = 10.8M | 768/12/12 = 124M |
| 激活 | ReLU | **GELU**（tanh 近似，对齐 OpenAI） |
| 命名 | 自己起的 | `transformer.wte/wpe/h/ln_f`，对齐 HF |

`Conv1D` 那几个权重要转置——HF 的 GPT-2 是从 TensorFlow 搬过来的，
`from_pretrained` 里那个 transpose 列表就是干这个的。


## 2 — 两个初始化细节

**权重共享（weight tying）**：`lm_head.weight = transformer.wte.weight`。
输入嵌入和输出投影用同一张表——省下 50257×768 ≈ **38M 参数**（占 124M 的 30%），
而且效果更好。07 章那个文件没做这件事（我核对过，两张表是独立的）。

**残差累积的缩放初始化**：残差流每经过一个子层方差就累加一次，
`n_layer` 层之后会涨到 `sqrt(n_layer)` 倍。所以投影层的 std 要乘 `(2*n_layer)**-0.5`
（`2` 是因为每个 Block 有两个子层）。GPT-2 论文里明确写了，
07 章的 `_init_weights` 只做了统一的 0.02。


## 3 — 速度优化（这一章最实用的部分）

**每加一项都单独测一次 tok/sec**，这是这一节的全部意义。四项累计约 10 倍。

| 优化 | 一句话 | 代价 |
|---|---|---|
| `torch.set_float32_matmul_precision('high')` | 用 TF32 做 matmul | 精度略降，几乎无损 |
| `torch.autocast(bfloat16)` | 激活用 bf16，参数仍 fp32 | 需要 Ampere+ |
| `torch.compile(model)` | 算子融合，减少显存往返 | 首次编译慢，和采样有冲突 |
| Flash Attention | `F.scaled_dot_product_attention` | 无，纯赚 |

**Flash Attention 我在你的 4060 上实测过**（384/6/6 配置，参数量完全相同）：
per-head 循环 7.0 ms/前向 → 融合 QKV + flash **3.3 ms**，2.1 倍。

**`vocab_size: 50257 -> 50304`** 是这一节最反直觉的一步：**增加**参数反而变快。
50304 = 128×393，是 2 的幂的倍数，能对齐 GPU 的 tile 边界；50257 是质数，
会走低效的 kernel 尾块。多出来的 47 个 token 永远不会被采到，
模型自己会把它们的 logit 学成 -inf。


## 4 — 超参数，逐项对齐 GPT-3 论文

GPT-2 论文没写清楚训练细节，所以 Karpathy 照 **GPT-3 论文**的附录来：

- AdamW `betas=(0.9, 0.95)`, `eps=1e-8`
- 梯度裁剪 `clip_grad_norm_(1.0)`
- **warmup + cosine 衰减**到 10%
- **weight decay 只给 2D 参数**（矩阵要，bias 和 LayerNorm 的 gain 不要）
- `fused=True` 的 AdamW（能用就用）
- **梯度累积**：总 batch 524288 tokens = 2^19，单卡装不下就累积
- DDP 多卡（你单卡，跳过，但代码要会读）

梯度累积有个必踩的坑：**loss 要除以累积步数**。`F.cross_entropy` 默认取 mean，
不除的话相当于把学习率放大了 `grad_accum_steps` 倍。


## 5 — 真实数据 + 评测

**FineWeb-Edu**（10B token 采样）替换 tiny shakespeare。`fineweb.py` 下载、
用 tiktoken 分词、存成 uint16 的 shard。约 20GB。

**HellaSwag** 常识推理评测。做法是把四个候选续写各算一遍 loss，取最低的那个——
一个纯 language model 不需要额外的分类头就能做选择题。
GPT-2 124M 的参考分数约 **29.5%**（随机是 25%）。

**验证集 + 定期采样**，看模型在学什么。


## 附：容易踩的地方

- **梯度累积时 loss 要 `/ grad_accum_steps`**，否则等于偷偷放大学习率
- `torch.compile` 和 `generate` 里的动态形状冲突——视频里为此把采样代码禁掉了
  （commit `4a63e6a` 的 message 就在吐槽这个）
- bf16 只在 Ampere 及以上可用；4060 是 Ada，没问题
- `model.require_backward_grad_sync` 在梯度累积 + DDP 下要手动管
- 4060 上 micro-batch 开太大会 OOM，先从 4 试
- FineWeb 下载中断能续，但 tokenize 是全量重来，建议 tmux / nohup
